In [1]:
import pandas as pd
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error, root_mean_squared_error
from sklearn.preprocessing import PolynomialFeatures
import numpy as np

In [2]:
concrete = pd.read_csv(r"Concrete_Data.csv")
concrete.head()

,Cement,Blast,Fly,Water,Superplasticizer,Coarse,Fine,Age,Strength
0,540.0,0.0,0.0,162.0,2.5,1040.0,676.0,28,79.99
1,540.0,0.0,0.0,162.0,2.5,1055.0,676.0,28,61.89
2,332.5,142.5,0.0,228.0,0.0,932.0,594.0,270,40.27
3,332.5,142.5,0.0,228.0,0.0,932.0,594.0,365,41.05
4,198.6,132.4,0.0,192.0,0.0,978.4,825.5,360,44.30


### Finding the best model

In [3]:
lr = LinearRegression()
X = concrete.drop("Strength", axis=1)
y = concrete["Strength"]

X_train,X_test,y_train,y_test=train_test_split(X,y , test_size=0.3, random_state=25)

lr.fit(X_train, y_train)
y_pred = lr.predict(X_test)

print(f"MAE: {mean_absolute_error(y_test, y_pred):.2f}")

MAE: 7.71


In [4]:
lr = LinearRegression()

poly = PolynomialFeatures(degree=2, include_bias=False).set_output(transform="pandas")
X_poly = poly.fit_transform(X)

X_train,X_test,y_train,y_test=train_test_split(X_poly,y , test_size=0.3, random_state=25)

lr.fit(X_train, y_train)
y_pred = lr.predict(X_test)

print(f"RMSE: {root_mean_squared_error(y_test, y_pred):.2f}")
print(f"MSE: {mean_squared_error(y_test, y_pred):.2f}")
print(f"MAE: {mean_absolute_error(y_test, y_pred):.2f}")
print(f"R2: {r2_score(y_test, y_pred):.2f}")


RMSE: 7.73
MSE: 59.79
MAE: 5.93
R2: 0.78


In [5]:
lr = LinearRegression()

poly = PolynomialFeatures(degree=3, include_bias=False).set_output(transform="pandas")
X_poly = poly.fit_transform(X)

X_train,X_test,y_train,y_test=train_test_split(X_poly,y , test_size=0.3, random_state=25)

lr.fit(X_train, y_train)
y_pred = lr.predict(X_test)

print(f"RMSE: {root_mean_squared_error(y_test, y_pred):.2f}")
print(f"MSE: {mean_squared_error(y_test, y_pred):.2f}")
print(f"MAE: {mean_absolute_error(y_test, y_pred):.2f}")
print(f"R2: {r2_score(y_test, y_pred):.2f}")

RMSE: 7.84
MSE: 61.54
MAE: 5.19
R2: 0.77


Degree 3 gives the best output

### Inferencing

In [6]:
#Prediction set
concretepredict = pd.read_csv(r"testConcrete.csv")

In [7]:
lr = LinearRegression()
poly = PolynomialFeatures(degree=3, include_bias=False).set_output(transform="pandas")

X_poly = poly.fit_transform(X)
C_poly = poly.transform(concretepredict)

# print(len(X_poly.columns))
lr.fit(X_poly, y)
y_pred = lr.predict(C_poly)

pd.concat([concretepredict, pd.Series(y_pred, name="Strength")], axis=1)

,Cement,Blast,Fly,Water,Superplasticizer,Coarse,Fine,Age,Strength
0,495,120,0,155,5,866,884,75,39.109814
1,262,129,0,271,2,808,787,174,-6.216227
2,201,48,1,215,5,807,839,113,107.152046
3,329,141,0,286,1,881,823,229,-165.571278
4,354,14,0,129,2,839,847,210,156.193649
5,150,23,23,114,4,883,638,36,1044.450490
6,480,64,0,292,3,896,776,180,110.161752
7,393,49,82,132,1,887,830,271,142.064565
8,284,63,1,138,1,804,725,44,342.428326
9,206,38,0,103,2,818,719,191,879.463657


In [8]:
import sklearn
sklearn.__version__

'1.7.2'

In [9]:
X = concrete.drop("Strength", axis=1)
y = concrete["Strength"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=25)

scores = []
for a in np.linspace(0.001, 1, 200):
    ridge = Ridge(alpha=a)
    ridge.fit(X_train, y_train)

    y_pred = ridge.predict(X_test)

    scores.append([a,
                   mean_absolute_error(y_test, y_pred),
                   root_mean_squared_error(y_test, y_pred),
                   r2_score(y_test, y_pred)])

df_scores = pd.DataFrame(scores, columns=["alpha", "MAE", "RMSE", "R^2"])
df_scores.sort_values(by="RMSE", inplace=True)
print(f"Min MAE, RSME, R^2 and corresponding alpha for Ridge Regression: \n{df_scores.iloc[0, :]}")


Min MAE, RSME, R^2 and corresponding alpha for Ridge Regression: 
alpha    0.001000
MAE      7.708183
RMSE     9.968421
R^2      0.635184
Name: 0, dtype: float64


In [10]:
X = concrete.drop("Strength", axis=1)
y = concrete["Strength"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=25)

scores = []
for a in np.linspace(0.001, 0.05, 200):
    lasso = Lasso(alpha=a)
    lasso.fit(X_train, y_train)

    y_pred = lasso.predict(X_test)

    scores.append([a,
                   mean_absolute_error(y_test, y_pred),
                   root_mean_squared_error(y_test, y_pred),
                   r2_score(y_test, y_pred)])

df_scores = pd.DataFrame(scores, columns=["alpha", "MAE", "RMSE", "R^2"])
df_scores.sort_values(by="RMSE", inplace=True)
print(f"Min MAE, RSME, R^2 and corresponding alpha for Lasso Regression: \n{df_scores.iloc[0,:]}")

Min MAE, RSME, R^2 and corresponding alpha for Lasso Regression: 
alpha    0.001000
MAE      7.708198
RMSE     9.968435
R^2      0.635183
Name: 0, dtype: float64


In [11]:
X = concrete.drop("Strength", axis=1)
y = concrete["Strength"]


X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=25)

scores = []
for a in np.linspace(0.001, 0.05, 20):
    for r in np.linspace(0.001, 1, 20):
        elastic = ElasticNet(alpha=a, l1_ratio=r)
        elastic.fit(X_train, y_train)

        y_pred = elastic.predict(X_test)

        scores.append([a,
                       r,
                       mean_absolute_error(y_test, y_pred),
                       root_mean_squared_error(y_test, y_pred),
                       r2_score(y_test, y_pred)])

df_scores = pd.DataFrame(scores, columns=["alpha", "l1Ratio", "MAE", "RMSE", "R^2"])
df_scores.sort_values(by="RMSE", inplace=True)
print(f"Min MAE, RSME, R^2 and corresponding alpha, l1Ratio for ElasticNet Regression: "
      f"\n{df_scores.iloc[0,:]}")

Min MAE, RSME, R^2 and corresponding alpha, l1Ratio for ElasticNet Regression: 
alpha      0.001000
l1Ratio    0.001000
MAE        7.708184
RMSE       9.968423
R^2        0.635184
Name: 0, dtype: float64
